# 03 — Scratch CNN, Pretrained ResNet18 and Grayscale

**Research questions:** Does ImageNet transfer learning improve MonReader page-turn classification across evaluation regimes? Does grayscale conversion change Scratch CNN generalization?

**Method:** Separate two-stage ImageNet-pretrained ResNet18 fitting for O/S/T/ST (5 head-only epochs; up to 20 fine-tuning epochs) and a controlled independently trained grayscale Scratch CNN comparison for O/ST. Use the frozen split manifests from Notebook 01. Do not transfer models between regimes or equate fresh metrics with recorded historical results.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPyImage
import src as mr
ROOT=Path.cwd().resolve()
settings=mr.load_settings(ROOT)
device=mr.setup(settings['seed'])
print('Device:',device,'| Two-stage pretrained ResNet18 enabled')
# Setting RERUN_HISTORICAL_MODELS = False reads the archived ResNet18, Scratch CNN
# and grayscale results without retraining or overwriting the historical evidence.
# Setting RERUN_HISTORICAL_MODELS = True retrains ResNet18 across O/S/T/ST
# and the O/ST grayscale comparators using the tracked src/ implementation.
RERUN_HISTORICAL_MODELS = False
print('ResNet18 mode:', 'optional new retraining' if RERUN_HISTORICAL_MODELS else 'accepted historic report')


## Original ResNet18 and colour-ablation evidence — reference only

Keep the originally reported held-out results visible while fresh model training runs independently. Original results must not be misrepresented as reproduced under a different split.

In [ ]:
historical_resnet=mr.read_archived_result_table(ROOT/'submission'/'results'/'resnet18_heldout_metrics.csv')
historical_colors=mr.read_archived_result_table(ROOT/'submission'/'results'/'final_color_ablation_summary.csv')
display(historical_resnet[['regime','n_samples','macro_f1','flip_f1']])
display(historical_colors)

## Two-stage ResNet18 comparison

For each regime, initialize ResNet18 with ImageNet weights and a new dropout/binary classification head. Optimize the head while the backbone is frozen for five epochs, then fine-tune the full network for at most 20 epochs using AdamW. Select the checkpoint using **validation Macro F1**. The original results are displayed by default; this optional code executes only when full rerun is explicitly enabled.


In [ ]:
if RERUN_HISTORICAL_MODELS:
    resnet=[]
    for regime in mr.REGIMES:
        split=pd.read_csv(Path(settings['out'])/'splits'/f'{regime}_split.csv')
        resnet.append(mr.train_resnet_regime(split,regime,settings,device))
        print(regime,'complete')
    rr=pd.DataFrame(resnet)
    rr.to_csv(Path(settings['out'])/'resnet18_heldout_metrics.csv',index=False)
    display(rr[['regime','val_macro_f1','test_macro_f1','test_fp','test_fn']])
else:
    rr=historical_resnet.rename(columns={'macro_f1':'test_macro_f1','accuracy':'test_accuracy'}).copy()
    rr['model_family']='resnet18'
    print('Accepted historical results; no new ResNet18 model trained.')
    display(rr[['regime','n_samples','test_macro_f1','test_accuracy']])


## Compare the two model families using the **same held-out metric**

Read the scratch results generated in Notebook 02 and the ResNet18 results calculated above. The bar chart uses held-out **Macro F1**, not W&B tuning F1, and is saved as an image suitable for the README. It makes no causal assertion about why transfer learning improves or worsens particular regimes.

In [ ]:
if RERUN_HISTORICAL_MODELS:
    scratch=pd.read_csv(Path(settings['out'])/'scratch_heldout_metrics.csv')
else:
    historic=mr.read_archived_result_table(ROOT/'submission'/'results'/'scratch_validation_test_metrics.csv')
    scratch=historic.loc[historic['split']=='Held-out Test'].rename(
        columns={'macro_f1':'test_macro_f1','accuracy':'test_accuracy'}).copy()
    scratch['model_family']='scratch'
comparison=pd.concat([scratch,rr],ignore_index=True)
display(comparison[['regime','model_family','test_macro_f1','test_accuracy']])
figure=mr.plot_model_comparison(comparison,Path(settings['out'])/'figures'/'model_comparison.png')
display(IPyImage(filename=str(figure)))


## Controlled grayscale experiment (Scratch CNN: O and ST)

The grayscale comparison is a **new training experiment** on corresponding train/validation/test partitions, not simply an RGB model evaluated on recoloured test images. Every grayscale model uses the selected RGB architecture and training settings for its regime, but starts with new random weights. Conversion is consistent across all three roles. This is a comparison of input representation, not a guarantee that performance will increase or decrease.

In [ ]:
if RERUN_HISTORICAL_MODELS:
    gray_results=[]
    for regime in ('O','ST'):
        df=pd.read_csv(Path(settings['out'])/'splits'/f'{regime}_split.csv')
        gray_results.append(mr.train_grayscale(df,regime,settings,device))
    gr=pd.DataFrame(gray_results);gr.to_csv(Path(settings['out'])/'grayscale_heldout_metrics.csv',index=False)
    display(gr[['regime','macro_f1','accuracy']])
    manifest=pd.read_csv(Path(settings['out'])/'canonical_manifest.csv')
    figure=mr.plot_grayscale_examples(manifest,Path(settings['out'])/'figures'/'rgb_vs_grayscale_examples.png')
    display(IPyImage(filename=str(figure)))
else:
    display(historical_colors)
    display(IPyImage(filename=str(ROOT/'figures'/'rgb_vs_grayscale_examples.png')))
    print('Grayscale comparison retained as original historical evidence.')


## Interpretation and handoff

In the accepted **historical report mode**, Notebook 03 displays the frozen model-comparison and grayscale results; it does not retrain the networks or regenerate all historical prediction files. The optional full rerun independently initializes/fine-tunes the model families per regime, saves per-frame predictions, and enables source-level uncertainty calculations in Notebook 04.

### Interpretation and operational relevance

Compare held-out Macro F1 and flip recall by regime rather than relying only on O, whose neighboring frames may be correlated across training and testing. Pretraining may improve representation reuse, but a source-safe test is essential for deployment on unfamiliar books, cameras, or users. Raw single-channel grayscale images require one third as many pixel-channel values as RGB, but that alone **does not establish one-third inference cost**; performance and actual pipeline latency need measurement.
